### Tabular datasets (DIF, IF and RECON)

#### 1. Credit card data from Kaggle

In [9]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from algorithms import DIF
from sklearn.ensemble import IsolationForest
import torch
import torch.nn as nn
import torch.optim as optim

# Load the data
df = pd.read_csv("new_data/tabular/creditcard.csv")

# Choose features
# FEATS = ['V3','V4','V10','V11','V12','V14','V16','V17']
FEATS = [c for c in df.columns if c not in ["Class", "Time"] and df[c].dtype != "O"]

# Sort by Time and then split by 80/20 
cut = int(len(df) * 0.8)
df_train, df_test = df.iloc[:cut], df.iloc[cut:]

# Use normal samples for training（Class=0）
df_train_norm = df_train[df_train["Class"] == 0].copy()

X_train = df_train_norm[FEATS].to_numpy(dtype=np.float32)
X_test  = df_test[FEATS].to_numpy(dtype=np.float32)
y_test  = df_test["Class"].to_numpy(dtype=np.int32)

# Standardlization
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

print(f"[DATA] Train(normals)={len(X_train)}, Test={len(X_test)}, Pos%={y_test.mean():.4%}, d={X_train.shape[1]}")


[DATA] Train(normals)=227428, Test=56962, Pos%=0.1317%, d=29


In [7]:
# DIF
n_ensemble = 50
n_estimators_per_repr = 6
TOTAL_TREES = n_ensemble * n_estimators_per_repr

dif = DIF(
    n_ensemble=n_ensemble,
    n_estimators=n_estimators_per_repr,  
    random_state=42
)
dif.fit(X_train)
score_dif = dif.decision_function(X_test)

auc_dif = roc_auc_score(y_test, score_dif)
pr_dif  = average_precision_score(y_test, score_dif)
print(f"[DIF]     AUC-ROC={auc_dif:.4f}  AUC-PR={pr_dif:.4f}  (Trees={TOTAL_TREES})")

network additional parameters: {'n_hidden': [500, 100], 'n_emb': 20, 'skip_connection': None, 'dropout': None, 'activation': 'tanh', 'be_size': 50}
[DIF]     AUC-ROC=0.9566  AUC-PR=0.2921  (Trees=300)


In [10]:
# IF
ifor = IsolationForest(
    n_estimators=TOTAL_TREES, 
    max_samples=256,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

ifor.fit(X_train)

score_if = -ifor.score_samples(X_test)

auc_if = roc_auc_score(y_test, score_if)
pr_if  = average_precision_score(y_test, score_if)
print(f"[IForest] AUC-ROC={auc_if:.4f}  AUC-PR={pr_if:.4f}  (Trees={TOTAL_TREES})")


[IForest] AUC-ROC=0.9493  AUC-PR=0.0349  (Trees=300)


In [11]:
# RECON
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class AE(nn.Module):
    def __init__(self, dim_in, dim_hid=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(dim_in, dim_hid), nn.ReLU(),
            nn.Linear(dim_hid, dim_hid // 2), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(dim_hid // 2, dim_hid), nn.ReLU(),
            nn.Linear(dim_hid, dim_in)
        )
    def forward(self, x): return self.decoder(self.encoder(x))

def run_recon(X_train, X_test, y_test, epochs=10, lr=1e-3):
    model = AE(X_train.shape[1]).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    Xt = torch.tensor(X_train, dtype=torch.float32, device=device)
    Xq = torch.tensor(X_test,  dtype=torch.float32, device=device)

    model.train()
    for ep in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(Xt), Xt)
        loss.backward()
        opt.step()
        if ep % 3 == 0:
            print(f"[RECON] epoch {ep:02d} loss {loss.item():.6f}")

    model.eval()
    with torch.no_grad():
        recon = model(Xq).cpu().numpy()

    score = np.mean((recon - X_test) ** 2, axis=1)
    auc = roc_auc_score(y_test, score)
    pr  = average_precision_score(y_test, score)
    return score, auc, pr

score_recon, auc_recon, pr_recon = run_recon(X_train, X_test, y_test)
print(f"[RECON] AUC-ROC={auc_recon:.4f}  AUC-PR={pr_recon:.4f}")

[RECON] epoch 00 loss 1.012887
[RECON] epoch 03 loss 1.006861
[RECON] epoch 06 loss 1.001211
[RECON] epoch 09 loss 0.995775
[RECON] AUC-ROC=0.9563  AUC-PR=0.0539


In [12]:
print("\n=== SUMMARY (Credit Card datasets) ====")
print(f"DIF     : AUC-ROC={auc_dif:.4f}  AUC-PR={pr_dif:.4f}")
print(f"IForest : AUC-ROC={auc_if:.4f}  AUC-PR={pr_if:.4f}")
print(f"RECON   : AUC-ROC={auc_recon:.4f}  AUC-PR={pr_recon:.4f}")


=== SUMMARY (Credit Card datasets) ====
DIF     : AUC-ROC=0.9566  AUC-PR=0.2921
IForest : AUC-ROC=0.9493  AUC-PR=0.0349
RECON   : AUC-ROC=0.9563  AUC-PR=0.0539


#### 2. Created datasets

In [ ]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from algorithms.dif import DIF
from sklearn.ensemble import IsolationForest
import torch
import torch.nn as nn
import torch.optim as optim


# Read created CSV（last column: label=0/1）
df = pd.read_csv("new_data/tabular/tabular_created.csv")
y = df["label"].to_numpy(dtype=int)
X = df.drop(columns=["label"]).to_numpy(dtype=float)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)

# DIF
model = DIF(n_ensemble=50, n_estimators=6)  
model.fit(X_train)

score = model.decision_function(X_test)     

# Verification direction
if y_test.mean()>0 and score[y_test==1].mean() < score[y_test==0].mean():
    score = -score

auc_dif = roc_auc_score(y_test, score)
pr_dif  = average_precision_score(y_test, score)
print("[DIF]", f"ROC-AUC={auc_dif:.4f}  PR-AUC={pr_dif:.4f}")


network additional parameters: {'n_hidden': [500, 100], 'n_emb': 20, 'skip_connection': None, 'dropout': None, 'activation': 'tanh', 'be_size': 50}
[DIF] ROC-AUC=0.8646  PR-AUC=0.2781


In [27]:
# IF
ifor = IsolationForest(
    n_estimators=300,
    max_samples=256,
    contamination='auto',
    random_state=0,
    n_jobs=-1
)
# Training
ifor.fit(X_train)

score_if = -ifor.score_samples(X_test)  
auc_if = roc_auc_score(y_test, score_if)
pr_if  = average_precision_score(y_test, score_if)
print(f"[IForest]  AUC-ROC={auc_if:.4f}  AUC-PR={pr_if:.4f}")

[IForest]  AUC-ROC=0.7027  AUC-PR=0.7078


In [33]:
# RECON
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class AE(nn.Module):
    def __init__(self, dim_in, dim_hid=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(dim_in, dim_hid),
            nn.ReLU(),
            nn.Linear(dim_hid, dim_hid // 2),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(dim_hid // 2, dim_hid),
            nn.ReLU(),
            nn.Linear(dim_hid, dim_in)
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

# Training
def train_autoencoder(X_train, X_test, y_test, n_epochs=10, lr=1e-3):
    model = AE(X_train.shape[1]).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

    for ep in range(n_epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(X_train_t), X_train_t)
        loss.backward()
        opt.step()
        if ep % 2 == 0:
            print(f"[Epoch {ep}] Train Loss = {loss.item():.6f}")

    
    model.eval()
    with torch.no_grad():
        recon = model(X_test_t).cpu().numpy()
    errors = np.mean((recon - X_test) ** 2, axis=1)

    auc_recon = roc_auc_score(y_test, errors)
    pr_recon = average_precision_score(y_test, errors)
    print(f"[RECON] AUC-ROC={auc_recon:.4f}, AUC-PR={pr_recon:.4f}")
    return errors, auc_recon, pr_recon
errors, auc_recon, pr_recon = train_autoencoder(X_train, X_test, y_test)


[Epoch 0] Train Loss = 1.015916
[Epoch 2] Train Loss = 1.011177
[Epoch 4] Train Loss = 1.006432
[Epoch 6] Train Loss = 1.001497
[Epoch 8] Train Loss = 0.996128
[RECON] AUC-ROC=0.7005, AUC-PR=0.7077


In [ ]:
print(f"\n===== SUMMARY (Created datasets) ======")
print(f"DIF     : AUC-ROC={auc_dif:.4f}  AUC-PR={pr_dif:.4f}")
print(f"IForest : AUC-ROC={auc_if:.4f}  AUC-PR={pr_if:.4f}")
print(f"RECON   : AUC-ROC={auc_recon:.4f}  AUC-PR={pr_recon:.4f}")


===== SUMMARY (Created datasets) ======
DIF     : AUC-ROC=0.8843  AUC-PR=0.3254
IForest : AUC-ROC=0.7027  AUC-PR=0.7078
RECON   : AUC-ROC=0.7005  AUC-PR=0.7077
